[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/exorbyte/mbox-cookbook/blob/main/05-explainability/01-index_types_and_recall_modes.ipynb)

In [1]:
# !pip install mbox

# Index Types and Recall Modes

Every notebook before this one asked you to trust that `index.match()` returns sensible, comparable scores. 

This notebook explains where those scores actually come from: M|BOX's two-phase design, the `IndexType` assigned to each column during indexing, and the `TableRecallMode` that decides how a query is compared against that column at search time.

In this notebook you will:

1. Recap the two-phase architecture, indexing and recall, and see exactly where each concept in this cookbook fits
2. Meet the six `IndexType` values, and see which one the SDK infers for each column in a real dataset
3. Confirm that every searchable `IndexType` defaults to a fuzzy, score-producing mode, not a strict yes/no check
4. Learn what a `0`-`100` score actually means: the portion of a reasonable match, not a percentage of shared characters
5. See that there are really only four comparison engines, string and numeric modes are the same four, reinterpreted
6. Get a map of which notebook in this directory to read next, based on the `IndexType` you actually have

## 1. Two phases: indexing, then recall

M|BOX splits every match into two separate steps, and keeping them separate is exactly what makes results explainable rather than a black box.

**Indexing** happens once, when you call `TableIndexer.create_index()`. Every column you index is assigned an `IndexType`, a decision about how that column's data is tokenized and stored. `TableIndexer` infers this automatically from your DataFrame, or you can declare it explicitly with `TableConfig` and `TableFieldConfig`, covered in `03-index-configuration/`.

**Recall** happens every time you call `.match()`. A `TableRecallMode` decides how your query is compared against the indexed values. `04-recall-tuning/01-understanding_match_modes.ipynb` already walked through what each mode does. What that notebook did not cover, and what this entire directory is about, is that `IndexType` and `TableRecallMode` are not independent choices. The `IndexType` you picked at indexing time changes what the *same* `TableRecallMode` actually does at recall time.

In [2]:
import pandas as pd
from mbox.indexing import TableIndexer

catalog = pd.read_csv("datasets/product_catalog.csv")
catalog

mbpie: 33 modules, 566 methods, 8 classes, 18 enums
  args: 426 required, 254 optional, 37 keywords, 39 flags, 26 arrays
  types: 372 int, 337 str, 1 double, 72 object


,product_id,product_name,category,year_released,price
0,B88-EXT-24,Extended Battery Pack Pro,Batteries,2024,39.99
1,A12-PWR-22,Portable Power Bank,Batteries,2022,29.50
2,C99-SNS-23,Motion Sensor Camera,Cameras,2023,89.00
3,D45-REL-21,Smart Relay Switch,Automation,2021,24.99
4,E67-LEN-24,Wide Angle Camera Lens,Cameras,2024,149.99
5,F31-TRK-20,GPS Tracker Module,Automation,2020,59.00
6,G14-CAB-19,Braided USB-C Cable,Accessories,2019,12.99
7,H82-DOC-23,Docking Station Hub,Accessories,2023,74.50
8,I29-BAT-22,Rechargeable Battery Cell,Batteries,2022,9.99
9,J56-SEN-24,Outdoor Motion Sensor,Automation,2024,44.00


This is the product catalog used throughout `05-explainability/`. It was built with one column for each `IndexType` this notebook covers: `product_id` is a structured code, `product_name` is free text, `category` is a single word, `year_released` is a whole number, `price` is a decimal.

## 2. `IndexType`: how a column is stored

`IndexType` is chosen once, per column, at indexing time. It determines how the engine tokenizes and stores that column's values, which in turn determines what a "reasonable match" even means for it later.

| IndexType Enum | Supported Data Dtypes | Best Used For |
|---|---|---|
| `IndexType.PHRASE` | String (`str`, `object`) | Full text, multi-word strings, street addresses, or multi-term descriptions containing spaces. |
| `IndexType.TERM` | String (`str`, `object`) | Single-word tokens, city names, country names, or single-term categories. |
| `IndexType.IDENT` | String (`str`, `object`) | Exact identifiers with minor variation tolerance (e.g. SKUs, product codes, tax IDs, passport numbers). |
| `IndexType.INTEGER` | Integer (`int`, `int64`) | Whole numbers, years, door numbers, postal codes. |
| `IndexType.DOUBLE` | Float / Numeric (`float`, `int`) | Floating-point values, prices, coordinates, ratings. |
| `IndexType.NON_SEARCHABLE` | Any | Display-only payload fields returned in results but excluded from search matching. |

You rarely have to pick these yourself. Build the index and ask it what it decided.

In [3]:
index = TableIndexer.create_index(
    catalog,
    index_columns=["product_id", "product_name", "category", "year_released", "price"],
    tmp_dir="tmp_index"
)

index.describe()

,Field,Index Type,Unique Value Count,Index size [bytes]
0,product_id,IndexType.IDENT,10,10972
1,product_name,IndexType.PHRASE,10,30024
2,category,IndexType.TERM,4,12676
3,year_released,IndexType.INTEGER,6,132
4,price,IndexType.DOUBLE,10,228


`TableIndexer` inferred `product_id` as `IDENT`, `product_name` as `PHRASE`, `category` as `TERM`, `year_released` as `INTEGER`, and `price` as `DOUBLE`, without you specifying anything. This is usually correct, and it is always inspectable: `describe()` tells you exactly what you are working with, rather than leaving it implicit.

## 3. Every searchable `IndexType` defaults to a fuzzy mode

Five of the six `IndexType` values are searchable, and by default, all five produce a graded `0`-`100` score rather than a strict pass or fail. For the three string types, that default is `APPROX`. For the two numeric types, it is the numeric equivalent. Let's confirm this directly: run each column with no `modes` argument at all, then run it again forcing the mode explicitly, and compare.

In [4]:
from mbox.recall import TableRecallMode

checks = [
    ("product_id", "B88-EXT-25"),        # one character off from B88-EXT-24
    ("product_name", "Extended Battery Pak Pro"),  # one typo
    ("category", "Batteries"),
    ("year_released", 2023),
    ("price", 40),
]

rows = []
for column, query in checks:
    default_result = index.match(**{column: query}, include_field_scores=True, min_total_match_value=0, max_results=1)
    approx_result = index.match(**{column: query}, modes={column: TableRecallMode.APPROX}, include_field_scores=True, min_total_match_value=0, max_results=1)
    rows.append({
        "column": column,
        "index_type": str(index.get_index_config().get_field(column).index_type),
        "default_mode_score": default_result[f"{column}_score"].iloc[0],
        "explicit_APPROX_score": approx_result[f"{column}_score"].iloc[0]
    })

pd.DataFrame(rows)

,column,index_type,default_mode_score,explicit_APPROX_score
0,product_id,IndexType.IDENT,81,81
1,product_name,IndexType.PHRASE,90,90
2,category,IndexType.TERM,100,100
3,year_released,IndexType.INTEGER,100,100
4,price,IndexType.DOUBLE,99,99


`default_mode_score` and `explicit_APPROX_score` match in every row. Whether the column is a string type or a numeric type, leaving `modes` unset gives you the same graded, typo- and distance-tolerant comparison you would get by asking for `APPROX` explicitly. This is why every earlier notebook in this cookbook could show typo-tolerant matching without ever mentioning `TableRecallMode`, it was `APPROX` the entire time, chosen for you by default.

## 4. Reading the score: the portion of a reasonable match

A score of `100` means the query is, for this `IndexType` and mode, indistinguishable from the indexed value. A score of `0` does not actually appear in your results under the defaults you have been using: matches that fall below the engine's internal cutoff are dropped entirely rather than returned with a low score, which `02-reading_field_level_scores.ipynb` covers in depth.

Everything in between `0` and `100` is **the portion of a reasonable match**: how much of what you would expect from a genuinely correct answer is actually present, given how this specific `IndexType` and mode define "expected." It is not a percentage of shared characters, and it is not a probability. A 20-point drop on a `PHRASE` field does not mean the same thing as a 20-point drop on a `TERM` field, because the two `IndexType`s define "reasonable" differently in the first place.

Here is that difference, directly. The same word, `"APPROXIMATELY"`, indexed three separate times under three different `IndexType`s, queried with an increasing number of substituted characters.

In [5]:
from mbox.config import TableConfig, TableFieldConfig, IndexType

def substitute_n(s, n, seed=0):
    """Return a copy of s with exactly n characters substituted, at fixed positions."""
    import random
    random.seed(seed)
    s = list(s)
    positions = random.sample(range(len(s)), n)
    for p in positions:
        s[p] = "Q" if s[p] != "Q" else "Z"
    return "".join(s)

def build_single_column_index(index_type):
    df = pd.DataFrame({"word": ["APPROXIMATELY"]})
    config = TableConfig(fields=[TableFieldConfig(column="word", index_type=index_type)])
    return TableIndexer.create_index(df=df, config_overrides=config, tmp_dir="tmp_index")

rows = []
for index_type in [IndexType.TERM, IndexType.IDENT, IndexType.PHRASE]:
    word_index = build_single_column_index(index_type)
    for n_edits in range(0, 6):
        query = substitute_n("APPROXIMATELY", n_edits) if n_edits else "APPROXIMATELY"
        result = word_index.match(word=query, modes={"word": TableRecallMode.APPROX}, include_field_scores=True, min_total_match_value=0)
        found = len(result) > 0 and result["index_row"].iloc[0] != -1
        rows.append({"index_type": index_type.name, "edits": n_edits, "word_score": result["word_score"].iloc[0] if found else None})

pd.DataFrame(rows).pivot(index="edits", columns="index_type", values="word_score")

config_overrides is given, therefore all information from index_columns, index_types, alias_sets and character_mappings is ignored.
config_overrides is given, therefore all information from index_columns, index_types, alias_sets and character_mappings is ignored.
config_overrides is given, therefore all information from index_columns, index_types, alias_sets and character_mappings is ignored.


index_type,IDENT,PHRASE,TERM
edits,,,
0,100.0,100.0,100.0
1,87.0,87.0,87.0
2,74.0,73.0,74.0
3,60.0,58.0,60.0
4,NaN,43.0,45.0
5,NaN,NaN,NaN


Same word, same query, same `APPROX` mode, three different outcomes. `TERM` and `PHRASE` both still find a match at 4 edits, `IDENT` has already dropped it. Where `TERM` and `PHRASE` do both still find a match, their scores are not identical, `74` versus `73`, `60` versus `58`, `IDENT`'s tolerance is deliberately tighter than the other two.

None of this is a bug or an inconsistency. `IDENT` exists specifically for identifiers, where you generally want the engine to be *less* forgiving than it would be for a product name or a free-text field, a near-miss on a tax ID or a SKU is a real difference, not a typo to be tolerated. `TERM` and `PHRASE` differ from each other for a more structural reason, covered in `03-phrase_fields_explained.ipynb` and `04-term_fields_explained.ipynb`: once a value has more than one word in it, the two `IndexType`s stop behaving similarly at all, not just slightly.

## 5. Where to go next

Each `IndexType` gets its own notebook, going through what every mode actually means for that specific kind of data, backed by real queries against the catalog above.

| Your column looks like... | `IndexType` | Notebook |
|---|---|---|
| A name, a description, an address, anything with spaces | `PHRASE` | `03-phrase_fields_explained.ipynb` |
| A single word: a category, a city, a status | `TERM` | `04-term_fields_explained.ipynb` |
| A code, a SKU, a tax ID, anything you'd call an identifier | `IDENT` | `05-ident_fields_explained.ipynb` |
| A year, a quantity, a price, a rating | `INTEGER` / `DOUBLE` | `06-numeric_fields_explained.ipynb` |

Before jumping to your specific `IndexType`, it's worth reading `02-reading_field_level_scores.ipynb` first, it covers exactly how `APPROX` arrives at a number, why matches disappear rather than fading to zero, and how `overall_score` combines multiple fields, all of which the per-type notebooks assume you already know.

## Next steps

- **`02-reading_field_level_scores.ipynb`** - the mechanics behind the scores you just saw: how `APPROX` computes a number, the hard cutoff, and the `overall_score` formula
- **`03-phrase_fields_explained.ipynb`** through **`06-numeric_fields_explained.ipynb`** - a deep dive per `IndexType`
- **`07-multi_field_weights_explained.ipynb`** - reading how several fields of *different* `IndexType`s combine into one `overall_score`
- **`08-debugging_a_bad_match.ipynb`** - putting all of it to work diagnosing a real, broken query

*M|BOX is currently in `beta`. Breaking changes may occur in minor releases until version `1.0.0`.*